In [6]:
import datetime
import pandas as pd
import requests
import os
import sys
from acw_api import AcwApi
import json
import time

with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)

In [57]:
class AcwApi:
    def __init__(self, prod=False):
        if prod is False:
            self.verify = False
            self.url = config['acw_setting']['dev_url']
        else:
            self.verify = True
            self.url = config['acw_setting']['prod_url']
        self.message_url = "messagecore/messages/"
        self.node_url = "tradecore/nodes/"
        self.kline_url = "infocore/kline/"

    def get_message(self, id=None):
        url = self.url + self.message_url
        if id is not None:
            fetch_list = False
            response = requests.get(url + str(id) + "/", verify=self.verify)
        else:
            fetch_list = True
            response = requests.get(url, verify=self.verify)
        if response.status_code == 200:
            if fetch_list:
                return pd.DataFrame(response.json()["results"])
            else:
                return pd.DataFrame([response.json()])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def create_message(self, telegram_chat_id, title, origin, type, content=None, remark=None, code=None, sent=False, send_times=1, send_term=1):
        url = self.url + self.message_url
        new_message_dict = {
            "datetime": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S"),
            "telegram_chat_id": telegram_chat_id,
            "title": title,
            "origin": origin,
            "type": type,
            "content": content,
            "remark": remark,
            "code": code,
            "sent": sent,
            "send_times": send_times,
            "send_term": send_term
        }
        response = requests.post(url, json=new_message_dict, verify=self.verify)
        if response.status_code == 201:
            return response.json()
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)

    def delete_message(self, id):
        url = self.url + self.message_url
        response = requests.delete(url + str(id) + "/", verify=self.verify)
        if response.status_code == 204:
            return True
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def get_node(self, id=None):
        url = self.url + self.node_url
        if id is not None:
            fetch_list = False
            response = requests.get(url + str(id) + "/", verify=self.verify)
        else:
            fetch_list = True
            response = requests.get(url, verify=self.verify)
        if response.status_code == 200:
            if fetch_list:
                return pd.DataFrame(response.json()["results"])
            else:
                return pd.DataFrame([response.json()])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def get_kline(self, target_market_code, origin_market_code, base_asset, interval, start_time=None, end_time=None):
        url = self.url + self.kline_url
        params = {
            "target_market_code": target_market_code,
            "origin_market_code": origin_market_code,
            "base_asset": base_asset,
            "interval": interval,
            "start_time": start_time.strftime("%Y-%m-%dT%H:%M:%S") if start_time is not None else None,
            "end_time": end_time.strftime("%Y-%m-%dT%H:%M:%S") if end_time is not None else None
        }
        response = requests.get(url, params=params, verify=self.verify)
        if response.status_code == 200:
            return pd.DataFrame(response.json())
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)


In [58]:
acw_api = AcwApi(prod=False)

In [67]:
start = time.time()
kline_df = acw_api.get_kline('UPBIT_SPOT/KRW', 'BINANCE_USD_M/USDT', 'BTC', '1T', start_time=datetime.datetime.now()+datetime.timedelta(hours=999), end_time=datetime.datetime.now()+datetime.timedelta(hours=0))
print(time.time()-start)

0.16115975379943848


/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [68]:
kline_df

,base_asset,datetime_now,tp_open,tp_high,tp_low,tp_close,LS_open,LS_high,LS_low,LS_close,SL_open,SL_high,SL_low,SL_close,dollar,tp,scr,atp24h,converted_tp,closed
0,BTC,2024-02-15T11:24:00+0000,2.485256,2.485256,2.439688,2.445420,2.484019,2.484019,2.441121,2.445420,2.482390,2.482390,2.439492,2.440925,1333.0,71485000.0,1.831935,6.055363e+11,69778619.60,True
1,BTC,2024-02-15T11:25:00+0000,2.445420,2.482582,2.438059,2.478735,2.445420,2.482778,2.441121,2.479385,2.440925,2.481148,2.436626,2.477755,1333.0,71480000.0,1.824812,6.058168e+11,69750080.56,True
2,BTC,2024-02-15T11:26:00+0000,2.480168,2.480168,2.433760,2.435388,2.460578,2.460578,2.435388,2.435388,2.458949,2.458949,2.433760,2.433760,1333.0,71478000.0,1.821963,6.060204e+11,69778619.60,True
3,BTC,2024-02-15T11:27:00+0000,2.435388,2.472969,2.407792,2.443329,2.435388,2.478979,2.409225,2.478979,2.433760,2.474483,2.407596,2.451553,1333.0,71486000.0,1.833359,6.065005e+11,69781020.08,True
4,BTC,2024-02-15T11:28:00+0000,2.443329,2.474679,2.401916,2.429980,2.478979,2.478979,2.402111,2.430176,2.451553,2.451553,2.400483,2.407059,1333.0,71501000.0,1.854727,6.066033e+11,69804758.16,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,BTC,2024-02-15T14:39:00+0000,2.585360,2.596286,2.480977,2.539181,2.586980,2.599879,2.484022,2.539376,2.556858,2.586855,2.482402,2.537755,1333.0,71901000.0,2.424536,5.700044e+11,70121354.80,True
196,BTC,2024-02-15T14:40:00+0000,2.537950,2.635605,2.480843,2.525526,2.540352,2.635605,2.481038,2.574121,2.538730,2.631058,2.477994,2.535460,1333.0,71999000.0,2.564139,5.705493e+11,70190435.28,True
197,BTC,2024-02-15T14:41:00+0000,2.576654,2.576654,2.494413,2.519849,2.576849,2.583281,2.500579,2.522892,2.538187,2.546042,2.479224,2.519849,1333.0,71997000.0,2.561290,5.707965e+11,70227376.00,True
198,BTC,2024-02-15T14:42:00+0000,2.519849,2.572111,2.487348,2.487348,2.522892,2.575290,2.487348,2.487348,2.519849,2.573670,2.485730,2.485730,1333.0,71996000.0,2.559866,5.713688e+11,70241778.88,True


In [12]:
response.json()

{'results': [{'uuid': '342b6270-cfd9-4676-90e4-125f67e4c03c',
   'email': 'superuser@halo-soft.net',
   'username': 'superuser',
   'first_name': 'Super',
   'last_name': 'User',
   'role': 'internal',
   'is_active': True,
   'date_joined': '2023-10-04T04:18:11+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': None,
    'user': '342b6270-cfd9-4676-90e4-125f67e4c03c'},
   'favorite_assets': [],
   'socialapps': [],
   'telegram_chat_id': None,
   'trade_config_allocations': []},
  {'uuid': '2c280bd2-6289-4e24-b1ee-5b7c5786265e',
   'email': 'ceo@halo-soft.net',
   'username': '헤일로테스트',
   'first_name': 'Chang-un',
   'last_name': 'Park',
   'role': 'user',
   'is_active': True,
   'date_joined': '2023-10-20T02:23:59+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': 'https://lh3.googleusercontent.com/a/ACg8ocIerr63_JO6LcR_Y4tWhz6pDV69zj8hMuvcc2eM_4fQ=s96-c',
    'user': '2c280bd2-6289-4e24-b1ee-5b7c5786265e'},
 

In [17]:
acw_api.get_node()

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-dev.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,id,name,url,description,max_user_count,market_code_services
0,1,trade_core_dev,http://221.148.128.205:20081,changun's test tradecore server,300,"[UPBIT_SPOT/KRW:BINANCE_USD_M/USDT, BITHUMB_SP..."


In [34]:
node = 'trade_core_dev'

all_node_df = acw_api.get_node()
node_df = all_node_df[all_node_df['name']==node]
if len(node_df) != 1:
    raise Exception(f"Node not found or not unique, len(node_df)={len(node_df)}")
node_df['market_code_services'].values[0]



/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-dev.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['UPBIT_SPOT/KRW:BINANCE_USD_M/USDT', 'BITHUMB_SPOT/KRW:BINANCE_SPOT/USDT']